In [25]:
import pandas as pd
import numpy as np
import polars as pl
import scipy.sparse as sp
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,f1_score
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
df = pl.read_parquet("../data/03_processed/baseline_features_train.parquet").to_pandas()
df = df.dropna(subset='IncidentGrade')
label_map = {
    "FalsePositive" : 0,
    'BenignPositive' : 1,
    'TruePositive' : 2
}
df['target'] = df['IncidentGrade'].map(label_map)


X = df.drop(columns=['IncidentGrade','start_time','end_time','target','OrgId','IncidentId'])
y = df['target']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_test.shape}")
print(f"Target distribution:\n{y_train.value_counts(normalize=True) * 100}")

Training set shape: (358277, 31)
Validation set shape: (89570, 31)
Target distribution:
target
1    48.575823
0    30.119433
2    21.304745
Name: proportion, dtype: float64


In [3]:
rf_baseline = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
print("Training the Random Forest Model ...")
rf_baseline.fit(X_train,y_train)

print("Predicting on validation set ...")
y_pred = rf_baseline.predict(X_test)

print("\n--- Classification Report ---")
target_labels = ['False Positive (0)','Benign Positive (1)','True Positive (2)']
print(classification_report(y_test,y_pred,target_names=target_labels))

macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f"\nCRITICAL BENCHMARK -> Baseline Macro-F1 Score: {macro_f1:.4f}")

Training the Random Forest Model ...
Predicting on validation set ...

--- Classification Report ---
                     precision    recall  f1-score   support

 False Positive (0)       0.81      0.60      0.69     26978
Benign Positive (1)       0.67      0.87      0.76     43509
  True Positive (2)       0.61      0.39      0.48     19083

           accuracy                           0.69     89570
          macro avg       0.70      0.62      0.64     89570
       weighted avg       0.70      0.69      0.68     89570


CRITICAL BENCHMARK -> Baseline Macro-F1 Score: 0.6430


## Baseline model on engineered features

In [4]:
df = pl.read_parquet("../data/03_processed/engineered_features_train.parquet").to_pandas()
df = df.dropna(subset='IncidentGrade')
df['target'] = df['IncidentGrade'].map(label_map)

X = df.drop(columns=['IncidentGrade','target','OrgId','IncidentId'])
y = df['target']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_test.shape}")
print(f"Target distribution:\n{y_train.value_counts(normalize=True) * 100}")

Training set shape: (358277, 37)
Validation set shape: (89570, 37)
Target distribution:
target
1    48.575823
0    30.119433
2    21.304745
Name: proportion, dtype: float64


In [5]:
rf_baseline = RandomForestClassifier(
    n_estimators = 50,
    max_depth=10,
    random_state=42,
    n_jobs = -1
)

print("Training the Random Forest Model ...")
rf_baseline.fit(X_train,y_train)

print("Predicting on validation set ...")
y_pred = rf_baseline.predict(X_test)

print("\n--- Classification Report ---")
print(classification_report(y_test,y_pred,target_names=target_labels))

macro_f1 = f1_score(y_test,y_pred,average='macro')
print(f"\nCRITICAL BENCHMARK -> Baseline Macro-F1 Score: {macro_f1:.4f}")

Training the Random Forest Model ...
Predicting on validation set ...

--- Classification Report ---
                     precision    recall  f1-score   support

 False Positive (0)       0.81      0.61      0.70     26978
Benign Positive (1)       0.67      0.87      0.76     43509
  True Positive (2)       0.62      0.40      0.49     19083

           accuracy                           0.69     89570
          macro avg       0.70      0.63      0.65     89570
       weighted avg       0.70      0.69      0.68     89570


CRITICAL BENCHMARK -> Baseline Macro-F1 Score: 0.6459


In [6]:
importance = pd.Series(rf_baseline.feature_importances_,index=X_train.columns)
print("Top 15 Most Important Features:")
print(importance.sort_values(ascending=False).head(15))

Top 15 Most Important Features:
evidence_per_second             0.089871
total_evidence_count            0.079487
unique_state_count              0.070856
unique_entitytype_count         0.067229
unique_countrycode_count        0.065187
unique_city_count               0.063227
is_multinational                0.061812
unique_sha256_count             0.038290
unique_filename_count           0.035781
unique_url_count                0.034859
ips_per_device                  0.032254
unique_accountobjectid_count    0.029955
unique_accountupn_count         0.029347
unique_applicationname_count    0.027688
unique_applicationid_count      0.027437
dtype: float64


## Version two of improved baseline model

In [7]:
df = pl.read_parquet('../data/03_processed/version_two_engineered_features_train.parquet').to_pandas()
df = df.dropna(subset=['IncidentGrade'])

df['target'] = df['IncidentGrade'].map(label_map)

X = df.drop(columns=['IncidentId','OrgId','IncidentGrade','target'])
y = df['target']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print(f"Shape of training data : {X_train.shape}")
print(f"Shape of test data : {X_test.shape}")
print(f"Target distribution in training data : {y_train.value_counts(normalize=True) * 100}")

Shape of training data : (358277, 57)
Shape of test data : (89570, 57)
Target distribution in training data : target
1    48.575823
0    30.119433
2    21.304745
Name: proportion, dtype: float64


In [11]:
rf_baseline = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs= -1 
)

print("Training the Random Forest Classifier ...")
rf_baseline.fit(X_train,y_train)

print("Predicting on Validation set ...")
y_pred = rf_baseline.predict(X_test)

print("\n --- Classification Report ---")
print(classification_report(y_test,y_pred,target_names=target_labels))

macro_f1 = f1_score(y_test,y_pred,average='macro')
print(f"\nCRITICAL BENCHMARK -> Baseline Macro-F1 Score: {macro_f1:.4f}")

Training the Random Forest Classifier ...


ValueError: could not convert string to float: '7973 22'

In [ ]:
importance = pd.Series(rf_baseline.feature_importances_,index = X_train.columns)
print("Top 15 Most Important Features:")
print(importance.sort_values(ascending=False).head(15))

Top 15 Most Important Features:
total_evidence_count          0.084056
evidence_per_second           0.078589
is_multinational              0.072003
unique_entitytype_count       0.064801
unique_state_count            0.062036
unique_countrycode_count      0.058722
unique_city_count             0.051875
unique_accountname_count      0.039121
devices_per_account           0.034301
cat_Execution                 0.029651
unique_url_count              0.029252
ips_per_device                0.028770
unique_filename_count         0.027619
unique_applicationid_count    0.027370
incident_duration_seconds     0.024246
dtype: float64


In [ ]:
print("Computing class weight for each class ...")
sample_weights = compute_sample_weight(class_weight='balanced' , y = y_train)

xgb_model = xgb.XGBClassifier(
    n_estimators  = 200,
    max_depth = 6,
    learning_rate = 0.1,
    objective = 'multi:softmax',
    num_class = 3,
    random_state = 42,
    n_jobs = -1
)


print('Training XGboost classiffier ...')
xgb_model.fit(X_train,y_train, sample_weight=sample_weights)

print('Predicting on validation set ...')
y_pred = xgb_model.predict(X_test)

print('\n --- Classification Report ---')
print(classification_report(y_test,y_pred,target_names=target_labels))

macro_f1 = f1_score(y_test,y_pred,average = 'macro')
print(f"\nCRITICAL BENCHMARK -> Baseline Macro-F1 Score: {macro_f1:.4f}")

importance = pd.Series(xgb_model.feature_importances_,index = X_train.columns)
print("Top 15 Most Important Features:")
print(importance.sort_values(ascending=False).head(15))

Computing class weight for each class ...
Training XGboost classiffier ...
Predicting on validation set ...

 --- Classification Report ---
                     precision    recall  f1-score   support

 False Positive (0)       0.74      0.68      0.71     26978
Benign Positive (1)       0.76      0.65      0.70     43509
  True Positive (2)       0.47      0.68      0.56     19083

           accuracy                           0.67     89570
          macro avg       0.66      0.67      0.66     89570
       weighted avg       0.69      0.67      0.67     89570


CRITICAL BENCHMARK -> Baseline Macro-F1 Score: 0.6564
Top 15 Most Important Features:
unique_countrycode_count        0.180754
unique_state_count              0.110351
cat_Execution                   0.072935
cat_Exfiltration                0.064844
cat_SuspiciousActivity          0.063744
unique_entitytype_count         0.051230
unique_applicationid_count      0.042063
evidence_per_second             0.033860
unique_detector

In [ ]:
df = pl.read_parquet("../data/03_processed/version_two_engineered_features_train.parquet").to_pandas()
df['text_AlertTitle'].head()

0                 2 3 278
1                  3659 2
2    1 12238 8823 6125 18
3                       0
4                     289
Name: text_AlertTitle, dtype: str

In [28]:
def space_tokenizer(x):
    return str(x).split()

In [29]:
X_base = df.drop(columns=['IncidentId','OrgId','IncidentGrade','target','text_AlertTitle','text_FileName'],errors='ignore')
y = df['target']
X_train_base,X_test_base,text_train_alert,text_test_alert,text_train_file,text_test_file,y_train,y_test = train_test_split(
    X_base,
    df['text_AlertTitle'].fillna(""),
    df['text_FileName'].fillna(""),
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Building TF-IDF Vocabularies...")

tfidf_alert = TfidfVectorizer(max_features=100,analyzer=space_tokenizer)
tfidf_file = TfidfVectorizer(max_features=100,analyzer=space_tokenizer)

train_alert_tfidf = tfidf_alert.fit_transform(text_train_alert)
test_alert_tfidf = tfidf_alert.transform(text_test_alert)

train_file_tfidf = tfidf_file.fit_transform(text_train_file)
test_file_tfidf = tfidf_file.transform(text_test_file)


print("Fusing Numeric and Text Matrices...")
X_train_fused = sp.hstack([sp.csr_matrix(X_train_base),train_alert_tfidf,train_file_tfidf])
X_test_fused = sp.hstack([sp.csr_matrix(X_test_base),test_alert_tfidf,test_file_tfidf])


print(f"Old training shape: {X_train_base.shape}")
print(f"New fused training shape: {X_train_fused.shape}")

print("\nComputing class weights...")
sample_weights = compute_sample_weight(class_weight='balanced',y=y_train)

print("Training XGBoost on Multi-Modal Data...")

xgb_fused = xgb.XGBClassifier(
    n_estimators = 200,
    max_depth = 6,
    learning_rate = 0.1,
    objective = 'multi : softmax',
    num_class = 3,
    random_state = 42,
    n_jobs = -1
)

xgb_fused.fit(X_train_fused,y_train,sample_weight=sample_weights)

print("\nPredicting on validation set...")
y_pred_fused = xgb_fused.predict(X_test_fused)

print('\n --- Fused NLP XGBoost Classification Report ---')
target_labels = ['FalsePositive (0)', 'BenignPositive (1)', 'TruePositive (2)']
print(classification_report(y_test, y_pred_fused, target_names=target_labels))

print(f"\nCRITICAL BENCHMARK -> Fused Macro-F1 Score: {f1_score(y_test, y_pred_fused, average='macro'):.4f}")

Building TF-IDF Vocabularies...
Fusing Numeric and Text Matrices...
Old training shape: (358277, 55)
New fused training shape: (358277, 255)

Computing class weights...
Training XGBoost on Multi-Modal Data...

Predicting on validation set...

 --- Fused NLP XGBoost Classification Report ---
                    precision    recall  f1-score   support

 FalsePositive (0)       0.72      0.73      0.73     26978
BenignPositive (1)       0.76      0.73      0.75     43509
  TruePositive (2)       0.57      0.60      0.58     19083

          accuracy                           0.70     89570
         macro avg       0.68      0.69      0.68     89570
      weighted avg       0.71      0.70      0.71     89570


CRITICAL BENCHMARK -> Fused Macro-F1 Score: 0.6849


In [30]:
y_probs = xgb_fused.predict_proba(X_test_fused)

tp_probs = y_probs[:, 2]

print(f"{'Threshold':<12}{'TP Precision':<15}{'TP Recall':<15}{'TP F1-Score':<15}")
print("-" * 55)

for threshold in np.arange(0.20, 0.75, 0.05):
    # If TP probability exceeds threshold, classify as 2; otherwise take max of class 0 and 1
    custom_preds = np.where(
        tp_probs >= threshold,
        2,
        np.argmax(y_probs[:, :2], axis=1) # choose between 0 and 1
    )
    
    prec = precision_score(y_test, custom_preds, labels=[2], average='macro', zero_division=0)
    rec = recall_score(y_test, custom_preds, labels=[2], average='macro', zero_division=0)
    f1 = f1_score(y_test, custom_preds, labels=[2], average='macro', zero_division=0)
    
    print(f"{threshold:<12.2f}{prec:<15.4f}{rec:<15.4f}{f1:<15.4f}")

Threshold   TP Precision   TP Recall      TP F1-Score    
-------------------------------------------------------
0.20        0.3275         0.9377         0.4855         
0.25        0.3582         0.9026         0.5129         
0.30        0.3957         0.8420         0.5383         
0.35        0.4828         0.7210         0.5784         
0.40        0.5424         0.6482         0.5906         
0.45        0.6186         0.5629         0.5894         
0.50        0.7024         0.4803         0.5705         
0.55        0.7826         0.4162         0.5434         
0.60        0.8312         0.3832         0.5246         
0.65        0.8430         0.3723         0.5165         
0.70        0.8495         0.3649         0.5105         


In [31]:
os.makedirs("../models", exist_ok=True)

print("Serializing artifacts to disk...")

joblib.dump(xgb_fused, "../models/xgb_fused_model.joblib")
print("Saved: xgb_fused_model.joblib")


joblib.dump(tfidf_alert, "../models/tfidf_alert_vectorizer.joblib")
print("Saved: tfidf_alert_vectorizer.joblib")


joblib.dump(tfidf_file, "../models/tfidf_file_vectorizer.joblib")
print("Saved: tfidf_file_vectorizer.joblib")

print("\nSerialization complete. All artifacts frozen.")

Serializing artifacts to disk...
Saved: xgb_fused_model.joblib
Saved: tfidf_alert_vectorizer.joblib
Saved: tfidf_file_vectorizer.joblib

Serialization complete. All artifacts frozen.


In [32]:
df_new = pd.read_csv("../data/04_predictions/scored_alerts.csv")
df_new

,IncidentId,Predicted_Threat_Level,True_Positive_Probability
0,21654,0,0.022977
1,293906,1,0.296630
2,428000,0,0.064164
3,147465,2,0.514679
4,417228,0,0.064164
...,...,...,...
236262,266339,1,0.133635
236263,323220,1,0.191245
236264,80720,1,0.178521
236265,465198,2,0.800687
